## cosmopedia-v2 data loading
- [ ] download HuggingFaceTB/smollm-corpus cosmopedia-v2
- [ ] reuse existing `vocab_fineweb-edu_32768.json` (no vocab training)
- [ ] encode + shuffle shards
- [ ] verify with `ShardDataset` from `toy_transformers.data`

In [ ]:
import sys
import os
import json
from pathlib import Path

def get_project_info() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    root = current
    for parent in [current, *current.parents]:
        if (parent / "toy_transformers").exists():
            root = parent
            break
    return root, current

if 'ROOT_DIR' not in globals():
    ROOT_DIR, EXPERIMENT_DIR = get_project_info()
    if str(ROOT_DIR) not in sys.path:
        sys.path.append(str(ROOT_DIR))
    if Path.cwd() != ROOT_DIR:
        os.chdir(ROOT_DIR)

from toy_transformers.tokenization import Vocabulary, bulk_encode, shuffle_shards
from toy_transformers.prepare_dataset import (
    get_shard_urls, download_shard,
    stream_texts,
    load_status, save_status, mark_phase_done, phase_done,
)
from toy_transformers.data import ShardDataset, S3Sync

print(f"ROOT_DIR:       {ROOT_DIR}")
print(f"EXPERIMENT_DIR: {EXPERIMENT_DIR}")

In [ ]:
# ── dataset / vocab config ──────────────────────────────────────────────────
DATASET_ID  = "HuggingFaceTB/smollm-corpus"
SUBSET      = "cosmopedia-v2"
SPLIT       = "train"

BOS         = "<BOS>"
PAD         = "<PAD>"

# reuse the fineweb-edu vocab — no retraining needed
VOCAB_PATH  = ROOT_DIR / "data/vocabs/vocab_fineweb-edu_32768.json"

# ── directory layout (mirrors prepare_dataset.py conventions) ───────────────
DATASET_DIR  = ROOT_DIR / "data/datasets/cosmopedia-v2"
RAW_DIR      = DATASET_DIR / "_raw"
ENCODED_DIR  = DATASET_DIR / "_encoded"
SHUFFLED_DIR = DATASET_DIR          # shuffled shards land in DATASET_DIR itself

SHARD_SIZE       = 128_000_000      # tokens per encoded shard
N_OUTPUT_SHARDS  = None             # None → same count as encoded
VAL_SHARDS       = 1
SEED             = 42

DATASET_DIR.mkdir(parents=True, exist_ok=True)

vocab = Vocabulary.load(VOCAB_PATH)
BOS_ID = vocab.token_to_idx[BOS.encode()]
PAD_ID = vocab.token_to_idx[PAD.encode()]
print(f"vocab size: {len(vocab.tokens)}, BOS={BOS_ID}, PAD={PAD_ID}")

In [ ]:
# ── download parquet shards ─────────────────────────────────────────────────
from tqdm import tqdm

status = load_status(DATASET_DIR)

if not phase_done(status, "download"):
    shard_urls = get_shard_urls(DATASET_ID, SUBSET, SPLIT)
    print(f"found {len(shard_urls)} shards")

    status["shard_urls"] = shard_urls
    status["num_raw_shards"] = len(shard_urls)
    save_status(DATASET_DIR, status)

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for i, url in tqdm(enumerate(shard_urls), total=len(shard_urls), desc="downloading"):
        dst = RAW_DIR / f"{i:04d}.parquet"
        download_shard(url, dst)

    mark_phase_done(DATASET_DIR, status, "download")
    print("download complete")
else:
    print("download already complete, skipping")

raw_files = sorted(RAW_DIR.glob("*.parquet"))
print(f"{len(raw_files)} parquet files in {RAW_DIR}")

In [ ]:
# ── encode with existing vocab ───────────────────────────────────────────────
# vocab_fineweb-edu_32768 is reused; no BPE training needed.

if not phase_done(status, "tokenize"):
    print(f"encoding {len(raw_files)} parquet files → {ENCODED_DIR}")
    ENCODED_DIR.mkdir(parents=True, exist_ok=True)

    bulk_encode(
        doc_iter=stream_texts(raw_files, BOS),
        vocab=vocab,
        vocab_path=VOCAB_PATH,
        output_dir=ENCODED_DIR,
        split_token=BOS,
        shard_size=SHARD_SIZE,
    )

    mark_phase_done(DATASET_DIR, status, "tokenize")
    print("encoding complete")
else:
    print("tokenize already complete, skipping")

with open(ENCODED_DIR / "metadata.json") as f:
    enc_meta = json.load(f)
print(f"{enc_meta['num_shards']} encoded shards, split_id={enc_meta['split_id']}")

In [ ]:
# ── shuffle shards ───────────────────────────────────────────────────────────
import numpy as np

if not phase_done(status, "shuffle"):
    n_out = N_OUTPUT_SHARDS or enc_meta["num_shards"]
    print(f"shuffling {enc_meta['num_shards']} encoded shards → {n_out} output shards")

    shuffle_shards(
        input_dir=ENCODED_DIR,
        output_dir=SHUFFLED_DIR,
        seed=SEED,
        n_output_shards=n_out,
        val_shards=VAL_SHARDS,
    )

    mark_phase_done(DATASET_DIR, status, "shuffle")
    print("shuffle complete")
else:
    print("shuffle already complete, skipping")

with open(SHUFFLED_DIR / "metadata.json") as f:
    meta = json.load(f)
print(f"train shards: {len(meta['train_shards'])}, val shard: {meta['val_shard']}")

In [ ]:
# ── verify a shuffled shard ──────────────────────────────────────────────────
from toy_transformers.tokenization import _read_shard

sample_name = meta["train_shards"][0]
shard = _read_shard(SHUFFLED_DIR / sample_name)
print(f"shard: {sample_name}  shape={shard.shape}  dtype={shard.dtype}")

bos_positions = (shard == BOS_ID).nonzero()[0]
print(f"BOS id: {BOS_ID}")
print(f"docs in shard: {len(bos_positions):,}")
print(f"first 5 BOS positions: {bos_positions[:5]}")

decoded = vocab.decode(shard[:300].tolist())
text = b"".join(decoded).decode("utf-8", errors="replace")
print(f"\n=== first 300 tokens decoded ===\n{text}")

In [ ]:
# ── test ShardDataset ────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader

BLOCK_SIZE = 1024

train_paths = [SHUFFLED_DIR / n for n in meta["train_shards"]]
val_paths   = [SHUFFLED_DIR / meta["val_shard"]]
print(f"train shards: {len(train_paths)}, val shards: {len(val_paths)}")

ds = ShardDataset(
    shard_paths=train_paths,
    block_size=BLOCK_SIZE,
    bos_id=BOS_ID,
    pad_id=PAD_ID,
    shuffle=True,
    seed=SEED,
)

loader = DataLoader(ds, batch_size=4, drop_last=True)

for i, (x, y, doc_ids, loss_mask) in enumerate(loader):
    print(f"batch {i}:")
    print(f"  x: {x.shape}  dtype={x.dtype}")
    for b in range(x.shape[0]):
        real   = loss_mask[b].sum().item()
        n_docs = (x[b] == BOS_ID).sum().item()
        print(f"  sample {b}: {real} real tokens, {BLOCK_SIZE - real} pad, {n_docs} docs")

    # shift sanity check
    real_len = loss_mask[0].sum().item()
    if real_len > 1:
        assert (y[0, :real_len - 1] == x[0, 1:real_len]).all(), "x/y shift mismatch!"
        print("  shift check passed")

    # decode first sample
    tokens = x[0, :min(100, real_len)].tolist()
    text   = b"".join(vocab.decode(tokens)).decode("utf-8", errors="replace")
    print(f"  first 100 tokens: {text[:200]}")

    if i >= 2:
        break

In [ ]:
# ── optional: upload to S3 ───────────────────────────────────────────────────
# Set S3_REMOTE to your bucket path to upload shuffled shards + vocab.
# Example: S3_REMOTE = "s3://my-bucket/toy-transformers"
S3_REMOTE = None

if S3_REMOTE:
    sync = S3Sync(remote_base=S3_REMOTE, local_root=ROOT_DIR)

    # upload shuffled shards
    for name in tqdm(meta["token_counts"], desc="uploading shards"):
        rel = (SHUFFLED_DIR / name).relative_to(ROOT_DIR)
        sync.push(rel)

    # upload metadata
    sync.push((SHUFFLED_DIR / "metadata.json").relative_to(ROOT_DIR))

    # upload vocab (already exists remotely from fineweb-edu, but push anyway)
    sync.push(VOCAB_PATH.relative_to(ROOT_DIR))

    print(f"upload complete → {S3_REMOTE}")
else:
    print("S3_REMOTE not set, skipping upload")